# Bringing Visualizations Together - An Introduction to Dash

This is the sixth notebook in the Plotly tutorial series

This notebook introduces [Dash](https://dash.plotly.com), a Python framework built on top of Plotly and Flask that allows you to turn your Plotly visualizations into interactive web applications. In this notebook, we will walk through two simple examples, taking a look at a scatter plot from the [the first tutorial](01_scatterplots_relationships.ipynb) and a violin plot from [the fourth tutorial](04_boxplots_distributions.ipynb) and build basic Dash apps around them. We'll also adding interactive components to show the strengths of Dash.

For these visualizations, we will be using the [Palmers Penguins](https://www.kaggle.com/datasets/satyajeetrai/palmer-penguins-dataset-for-eda) dataset, found in the [data directory](../data)

In [1]:
# Imports
import numpy as np
import plotly.graph_objects as go
import statsmodels
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd

First, let's load in the dataset

In [2]:
# Load in data
df = pd.read_csv('../data/penguins.csv')
df.dropna(inplace = True)
df.head()

,id,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
4,4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007


Now, let's initialize the Dash app and create a basic interactive web app

In [3]:
app = dash.Dash(__name__)

species_options = [
    {'label': species, 'value': species} for species in df['species'].dropna().unique()
]

app.layout = html.Div([
    html.H1("Penguins Dashboard"),

    html.H2("Scatter Plot Viewer"),
    html.Label("Select Species:"),
    dcc.Dropdown(
        id='species-dropdown',
        options=species_options,
        value='Adelie',
        clearable=False
    ),
    dcc.Graph(id='scatter-plot'),

    html.H2("Flipper Length Violin Plot"),
    html.Label("Select Species:"),
    dcc.Dropdown(
        id='violin-species-dropdown',
        options=species_options,
        value='Adelie',
        clearable=False
    ),
    dcc.Checklist(
        id='display-options',
        options=[
            {'label': 'Show Box Plot', 'value': 'box'},
            {'label': 'Show All Points', 'value': 'points'}
        ],
        value=['box', 'points'],
        inline=True
    ),
    dcc.Graph(id='violin-plot')
])

# Add interactivity
@app.callback(
    Output('scatter-plot', 'figure'),
    Input('species-dropdown', 'value')
)
def update_scatter(selected_species):
    filtered_df = df[df['species'] == selected_species]
    fig = px.scatter(
        filtered_df,
        x="bill_length_mm",
        y="bill_depth_mm",
        color="island",
        symbol="sex",
        title=f"Bill Dimensions of {selected_species} Penguins"
    )
    return fig


@app.callback(
    Output('violin-plot', 'figure'),
    Input('violin-species-dropdown', 'value'),
    Input('display-options', 'value')
)
def update_violin(selected_species, display_opts):
    box = 'box' in display_opts
    points = 'all' if 'points' in display_opts else False

    filtered_df = df[df['species'] == selected_species]

    fig = px.violin(
        filtered_df,
        x="species",
        y="flipper_length_mm",
        color="sex",
        box=box,
        points=points,
        title="Flipper Length Distribution by Species",
        labels={
            'flipper_length_mm': 'Flipper Length (mm)',
            'species': 'Species',
            'sex': 'Sex'
        },
        width=1000,
        height=600
    )
    return fig

# Run the app
if __name__ == '__main__':
     app.run_server(debug=True)

## Final Remarks

Dash allows you to combine the analytical clarity of Plotly visualizations with the accessibility of web-based interactivity. This notebook showcased how multiple Plotly figures can be integrated into a single dashboard. This is related to Tufte's idea of *small multiples* where we can put varying analysis of the same dataset next to each other to allow for viewers to see the changes/differences we want them to see. The interactive component also increases the *data density*, since we can pack very high dimensional data into the interactive components of the plot.

More broadly, throughout this tutorial series, we explored key types of visualizations:
- Scatter plots for analyzing relationships
- Line and stacked area charts for visualizing change over time
- Bar charts and radar plots for comparing categories
- Histograms, box plots, and violin plots for examining distributions
- Bubble charts and parallel coordinates for understanding multivariate and high-dimensional data

Each notebook emphasized not only how to generate these visualizations using Plotly, but also why and when to use them. Overall, these tutorials should serve as a guide to designing clear, meaningful and interactive visualizations using Plotly. Hopefully, through the references these introductory frameworks and references thoughout the notebooks, you will be able to use this tool to create meaningful visualizations of your own.

Thanks for following along!

## References
- [Plotly Dash Documentation](https://dash.plotly.com/)
- *The Visual Display of Quantitative Information*, Edward Tufte

